In [1]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *

In [2]:
df_tabela_consolidada = pd.read_csv(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/data/tabela_consolidada.csv")
df_tabela_consolidada

,Data_UTC,btc_rvi_diff,btc_hv_log_ret,btc_funding_rate_mean,btc_funding_rate_diff,btc_miner_net_pos_change,btc_exchange_supply_diff,btc_log_ret,btc_spx_corr_30d,btc_coinbase_premium_diff,...,nasdaq_log_ret,gold_log_ret,us10y_diff,vix_log_ret,total3_log_ret,ssr_diff,flippening_ratio_diff,social_vol_log_spread,social_momentum,total_market_noise_z
0,2016-12-31,NaN,NaN,-0.000421,-0.000324,NaN,NaN,NaN,-0.162835,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.220099,0.000000,0.000000
1,2017-01-01,4.586413,-0.010119,0.001116,0.001537,0.000938,0.028501,0.026343,-0.162835,NaN,...,NaN,NaN,NaN,NaN,0.006530,-0.387567,NaN,-3.449948,-0.229849,0.000000
2,2017-01-02,4.467393,-0.226495,0.001892,0.000776,0.000732,0.034168,0.019024,-0.162835,NaN,...,NaN,NaN,NaN,NaN,0.039637,0.029452,NaN,-3.382149,0.067799,0.000000
3,2017-01-03,3.898178,-0.166424,0.001352,-0.000540,-0.002485,-0.012473,0.016387,-0.162835,NaN,...,0.003349,0.004217,NaN,NaN,0.020670,0.022461,NaN,-3.771278,-0.389129,0.000000
4,2017-01-04,4.286328,0.447093,0.001414,0.000062,0.016571,0.240520,0.090700,-0.162835,NaN,...,0.006199,0.014325,-0.0059,-0.074941,0.079352,0.117991,NaN,-3.696498,0.074780,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3393,2025-07-28,-4.828541,0.012820,0.000055,-0.000003,-0.000576,-0.013170,-0.011507,0.257893,-41.70,...,-0.002303,0.004040,-0.0900,0.007509,-0.039505,-0.010777,-0.001599,-5.923719,-0.491453,0.326016
3394,2025-07-29,-4.671673,0.000497,0.000069,0.000014,0.000387,-0.000905,-0.001223,0.270594,-25.64,...,0.011094,-0.015534,0.0480,0.056361,-0.007834,-0.001299,-0.000005,-5.665960,0.257759,-0.295265
3395,2025-07-30,-4.259905,-0.013026,0.000055,-0.000014,0.000665,0.001612,-0.001066,0.332494,7.22,...,-0.018605,0.004267,0.0100,-0.029907,-0.015516,-0.002737,0.000999,-5.659326,0.006634,-0.281078
3396,2025-07-31,-5.448235,0.129060,0.000041,-0.000014,-0.001344,-0.006516,-0.017592,0.364173,7.20,...,-0.017924,0.021884,-0.1660,0.072589,-0.027866,-0.018982,-0.002337,-5.428189,0.231137,-0.826857


In [3]:
def verificar_estacionaridade(df, nivel_significancia=0.05):
    """
    Roda o teste Augmented Dickey-Fuller (ADF) em todas as colunas numéricas.
    Retorna um DataFrame com os resultados.
    """
    resultados = []
    
    print(f"Iniciando Teste ADF em {len(df.columns)} variáveis...")
    print("-" * 80)
    
    # Selecionar apenas colunas numéricas (ignorar Data_UTC se estiver como coluna)
    cols_numericas = df.select_dtypes(include=[np.number]).columns
    
    for col in cols_numericas:
        # Limpeza rápida para o teste (ADF não aceita NaNs ou Infinitos)
        serie_limpa = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(serie_limpa) < 20: # Precisa de dados mínimos
            resultados.append({
                'Variavel': col,
                'ADF Statistic': np.nan,
                'P-Valor': np.nan,
                'Resultado': 'Dados Insuficientes'
            })
            continue

        try:
            # Rodando o Teste ADF
            result = adfuller(serie_limpa, autolag='AIC')
            
            p_value = result[1]
            adf_stat = result[0]
            
            # Critério de Decisão
            if p_value < nivel_significancia:
                status = "✅ ESTACIONÁRIA"
            else:
                status = "❌ NÃO ESTACIONÁRIA"
                
            resultados.append({
                'Variavel': col,
                'ADF Statistic': round(adf_stat, 4),
                'P-Valor': round(p_value, 6),
                'Resultado': status
            })
            
        except Exception as e:
            resultados.append({
                'Variavel': col,
                'ADF Statistic': np.nan,
                'P-Valor': np.nan,
                'Resultado': f"Erro: {str(e)}"
            })

    # Criando DataFrame do Relatório
    df_resultado = pd.DataFrame(resultados)
    df_resultado = df_resultado.sort_values(by='P-Valor', ascending=True)
    
    return df_resultado

## 📊 Interpretação dos Resultados do Teste ADF

### ✅ Sinal Verde (ESTACIONÁRIA)

**O que significa:**  
O P-Valor foi menor que 0.05 (ex: 0.000000).

**Ação:**  
Essa variável está perfeita. Ela entra direto no modelo VAR ou Teste de Granger.

**Exemplo Esperado:**  
`btc_log_ret`, `us10y_diff`, `eth_vol_log_ret`. Como aplicamos Log e Diff, 99% das suas features devem cair aqui.

---

### ❌ Sinal Vermelho (NÃO ESTACIONÁRIA)

**O que significa:**  
O P-Valor foi alto (ex: 0.85 ou 0.15). A variável ainda tem tendência (sobe ou desce com o tempo).

**Causa provável:**  
Talvez alguma feature tenha escapado do tratamento (ex: se você acidentalmente colocou o Preço Bruto `btc_price` em vez do retorno).

**Ação:**

- **Se for uma variável bruta (ex: Preço):** Não use no teste de Granger. Use a versão tratada (`_log_ret` ou `_diff`).
  
- **Se for uma variável já tratada (ex: `_diff`) que falhou:** Tente aplicar uma segunda diferença (`.diff().diff()`).

In [ ]:
df_adf_report = verificar_estacionaridade(df_tabela_consolidada, nivel_significancia=0.05)
# df_adf_report.to_excel(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/src/output/estacionaridade_report.xlsx", index=False)
df_adf_report

Iniciando Teste ADF em 29 variáveis...
--------------------------------------------------------------------------------


,Variavel,ADF Statistic,P-Valor,Resultado
0,btc_rvi_diff,-15.8612,0.000000,✅ ESTACIONÁRIA
24,flippening_ratio_diff,-12.5488,0.000000,✅ ESTACIONÁRIA
23,ssr_diff,-59.1435,0.000000,✅ ESTACIONÁRIA
22,total3_log_ret,-23.0338,0.000000,✅ ESTACIONÁRIA
21,vix_log_ret,-17.6953,0.000000,✅ ESTACIONÁRIA
20,us10y_diff,-16.7799,0.000000,✅ ESTACIONÁRIA
19,gold_log_ret,-22.6144,0.000000,✅ ESTACIONÁRIA
18,nasdaq_log_ret,-13.0065,0.000000,✅ ESTACIONÁRIA
17,dxy_log_ret,-53.4106,0.000000,✅ ESTACIONÁRIA
16,spx_log_ret,-13.2854,0.000000,✅ ESTACIONÁRIA
